# Лабораторная работа №2
## Современные CV-архитектуры: ConvNeXt vs ViT и самообучение (MAE)

Цель: освоить современные архитектуры компьютерного зрения (CNN нового поколения и Vision Transformers), понять принципы self-supervised learning через Masked Autoencoders, научиться анализировать ошибки классификации.

In [ ]:
# === УСТАНОВКА ПАКЕТОВ ДЛЯ COLAB ===
# Выполнить в начале каждой лабораторной

# Для ЛР2 (timm для современных CV-моделей)
!pip install -q timm

# Проверка GPU
import torch
assert torch.cuda.is_available(), "В Colab: Runtime -> Change runtime type -> GPU"
print(f"GPU доступен: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# ЛАБОРАТОРНАЯ РАБОТА №2
# Современные архитектуры Computer Vision и самообучение
# ============================================================

# === ИМПОРТЫ ===
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T
from torchvision import datasets
import timm
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import time

import os
import copy
import math
import pickle
import random
import warnings
import pandas as pd
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from torch.utils.data import Subset
from torch.utils.flop_counter import FlopCounterMode
from sklearn.linear_model import LogisticRegressionCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from matplotlib.colors import PowerNorm
from IPython.display import display, Markdown

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
torch.backends.cudnn.benchmark = True
USE_AMP = device.type == 'cuda'
warnings.filterwarnings('ignore', message='Detected call of `lr_scheduler.step()`')
warnings.filterwarnings('ignore', category=FutureWarning, module='sklearn')


def autocast():
    return torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP)


def sync():
    if device.type == 'cuda':
        torch.cuda.synchronize()


def count_params(model):
    return sum(p.numel() for p in model.parameters())


plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3, 'lines.linewidth': 2})
BLUE, ORANGE, AQUA = '#2a78d6', '#eb6834', '#1baf7a'

In [ ]:
# === ПАРАМЕТРЫ ===
DATA_DIR = './data'
CKPT_DIR = './checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
NUM_WORKERS = 2

MODEL_NAMES = ['convnext_tiny', 'vit_small_patch16_224']
IMG_SIZE = 224
FT_EPOCHS = 5
FT_BATCH = 128
FT_LR = 1e-4
FT_WD = 0.05
VAL_SIZE = 5000
ROBUST_N = 2000

MAE_IMG = 128
MAE_PATCH = 16
MAE_MASK_RATIO = 0.75
MAE_EPOCHS = 60
MAE_BATCH = 256
MAE_LR = 5e-4
MAE_WARMUP = 5
ENC_DIM, ENC_DEPTH, ENC_HEADS = 192, 8, 3
DEC_DIM, DEC_DEPTH, DEC_HEADS = 128, 4, 4
LABELED_PER_CLASS = 5
SMALL_EPOCHS = 200
SMALL_LR = 5e-4

In [ ]:
# ============================================================
# ЧАСТЬ 1. ConvNeXt vs Vision Transformer на CIFAR-100
# ============================================================
# ЗАДАНИЕ:
# 1. Загрузить CIFAR-100 (100 классов, 32x32)
# 2. Обучить ConvNeXt-Tiny и ViT-Small
# 3. Сравнить: accuracy, params, FLOPs, latency, robustness
# ============================================================

# === РЕШЕНИЕ ===

# --- 1.1 Загрузка CIFAR-100 ---
train_raw = datasets.CIFAR100(DATA_DIR, train=True, download=True)
test_raw = datasets.CIFAR100(DATA_DIR, train=False, download=True)
CLASSES = train_raw.classes
NUM_CLASSES = len(CLASSES)
y_train_np = np.array(train_raw.targets)
y_test_np = np.array(test_raw.targets)

with open(os.path.join(DATA_DIR, 'cifar-100-python', 'meta'), 'rb') as f:
    COARSE_NAMES = pickle.load(f, encoding='latin1')['coarse_label_names']
with open(os.path.join(DATA_DIR, 'cifar-100-python', 'train'), 'rb') as f:
    _d = pickle.load(f, encoding='latin1')
FINE_TO_COARSE = np.zeros(NUM_CLASSES, dtype=int)
FINE_TO_COARSE[np.array(_d['fine_labels'])] = np.array(_d['coarse_labels'])
print(f"train: {train_raw.data.shape}, test: {test_raw.data.shape}, классов: {NUM_CLASSES}, суперклассов: {len(COARSE_NAMES)}")

perm = np.random.default_rng(SEED).permutation(len(train_raw))
val_idx, train_idx = perm[:VAL_SIZE], perm[VAL_SIZE:]
print(f"train: {len(train_idx)}, val: {len(val_idx)}, test: {len(test_raw)}")

fig, axes = plt.subplots(2, 10, figsize=(16, 3.8))
for ax, c in zip(axes.ravel(), np.random.default_rng(0).choice(NUM_CLASSES, 20, replace=False)):
    i = np.where(y_train_np == c)[0][0]
    ax.imshow(train_raw.data[i], interpolation='bicubic')
    ax.set_title(CLASSES[c], fontsize=9)
    ax.axis('off')
fig.suptitle('Примеры CIFAR-100 (32x32)')
plt.tight_layout()
plt.show()

In [ ]:
# --- 1.2 Аугментации и загрузчики ---
def make_transforms(mean, std):
    train_tf = T.Compose([
        T.Resize(IMG_SIZE),
        T.RandomCrop(IMG_SIZE, padding=IMG_SIZE // 14, padding_mode='reflect'),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
        T.Normalize(mean, std),
    ])
    test_tf = T.Compose([T.Resize(IMG_SIZE), T.ToTensor(), T.Normalize(mean, std)])
    return train_tf, test_tf


def make_loaders(mean, std):
    train_tf, test_tf = make_transforms(mean, std)
    train_ds = Subset(datasets.CIFAR100(DATA_DIR, train=True, transform=train_tf), train_idx)
    val_ds = Subset(datasets.CIFAR100(DATA_DIR, train=True, transform=test_tf), val_idx)
    test_ds = datasets.CIFAR100(DATA_DIR, train=False, transform=test_tf)
    kw = dict(num_workers=NUM_WORKERS, pin_memory=device.type == 'cuda')
    gen = torch.Generator().manual_seed(SEED)
    return (DataLoader(train_ds, FT_BATCH, shuffle=True, drop_last=True, generator=gen, **kw),
            DataLoader(val_ds, 256, shuffle=False, **kw),
            DataLoader(test_ds, 256, shuffle=False, **kw))

In [ ]:
# --- 1.3 Модели, обучение, предсказание ---
def load_model(name):
    model = timm.create_model(name, pretrained=True, num_classes=NUM_CLASSES)
    cfg = timm.data.resolve_model_data_config(model)
    return model.to(device).to(memory_format=torch.channels_last), cfg


@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs, labels = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last)
        with autocast():
            logits = model(x)
        probs.append(logits.float().softmax(1).cpu())
        labels.append(y)
    return torch.cat(probs).numpy(), torch.cat(labels).numpy()


def train_one_epoch(model, loader, opt, sched, scaler, criterion):
    model.train()
    loss_sum = torch.zeros((), device=device)
    correct = torch.zeros((), device=device)
    n = 0
    for x, y in loader:
        x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last)
        y = y.to(device, non_blocking=True)
        with autocast():
            logits = model(x)
            loss = criterion(logits, y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        sched.step()
        loss_sum += loss.detach() * len(y)
        correct += (logits.argmax(1) == y).sum()
        n += len(y)
    return loss_sum.item() / n, correct.item() / n


def finetune(name):
    model, cfg = load_model(name)
    loaders = make_loaders(cfg['mean'], cfg['std'])
    ckpt = os.path.join(CKPT_DIR, f'{name}_cifar100_{FT_EPOCHS}ep.pt')
    if os.path.exists(ckpt):
        state = torch.load(ckpt, map_location=device)
        model.load_state_dict(state['model'])
        print(f"{name}: загружен чекпоинт {ckpt}")
        return model, cfg, loaders, state['hist']

    train_loader, val_loader, _ = loaders
    opt = optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=FT_WD)
    sched = optim.lr_scheduler.OneCycleLR(opt, max_lr=FT_LR, total_steps=FT_EPOCHS * len(train_loader),
                                          pct_start=0.1, anneal_strategy='cos')
    scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    hist = {'train_loss': [], 'train_acc': [], 'val_acc': [], 'epoch_time': []}
    for epoch in range(FT_EPOCHS):
        sync()
        t0 = time.perf_counter()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, opt, sched, scaler, criterion)
        sync()
        dt = time.perf_counter() - t0
        vp, vy = predict(model, val_loader)
        va = float((vp.argmax(1) == vy).mean())
        for k, v in zip(hist, (tr_loss, tr_acc, va, dt)):
            hist[k].append(v)
        print(f"{name} | эпоха {epoch + 1}/{FT_EPOCHS} | train loss {tr_loss:.3f} | train acc {tr_acc:.3f} | "
              f"val acc {va:.3f} | {dt:.0f} c")
    torch.save({'model': model.state_dict(), 'hist': hist}, ckpt)
    return model, cfg, loaders, hist

In [ ]:
# --- 1.4 Дообучение обеих моделей ---
results = {}
for name in MODEL_NAMES:
    set_seed(SEED)
    model, cfg, loaders, hist = finetune(name)
    probs, labels = predict(model, loaders[2])
    top5 = np.argsort(-probs, axis=1)[:, :5]
    results[name] = dict(model=model, cfg=cfg, hist=hist, probs=probs, labels=labels,
                         test_acc=float((probs.argmax(1) == labels).mean()),
                         test_top5=float((top5 == labels[:, None]).any(1).mean()))
    print(f"{name}: test acc = {results[name]['test_acc']:.4f}, top-5 = {results[name]['test_top5']:.4f}\n")
    if device.type == 'cuda':
        torch.cuda.empty_cache()

In [ ]:
# --- 1.5 Кривые обучения ---
SHORT = {'convnext_tiny': 'ConvNeXt-T', 'vit_small_patch16_224': 'ViT-S/16'}
COLORS = {'convnext_tiny': BLUE, 'vit_small_patch16_224': ORANGE}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for name, r in results.items():
    ep = np.arange(1, len(r['hist']['train_loss']) + 1)
    axes[0].plot(ep, r['hist']['train_loss'], 'o-', color=COLORS[name], label=SHORT[name])
    axes[1].plot(ep, r['hist']['train_acc'], 'o--', color=COLORS[name], lw=1.5, label=f'{SHORT[name]} train')
    axes[1].plot(ep, r['hist']['val_acc'], 'o-', color=COLORS[name], label=f'{SHORT[name]} val')
    axes[2].bar(np.array(ep) + (0.2 if name == MODEL_NAMES[1] else -0.2), r['hist']['epoch_time'],
                width=0.4, color=COLORS[name], label=SHORT[name])
axes[0].set_title('Train loss')
axes[1].set_title('Accuracy: train (пунктир) и val')
axes[2].set_title('Время эпохи, с')
for ax in axes:
    ax.set_xlabel('эпоха')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# --- 1.6 Параметры, FLOPs, latency ---
@torch.no_grad()
def count_gflops(model):
    model.eval()
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
    with FlopCounterMode(display=False) as fc:
        model(x)
    return fc.get_total_flops() / 1e9


@torch.inference_mode()
def measure_latency(model, batch_size, n_warmup=10, n_iters=50):
    model.eval()
    x = torch.randn(batch_size, 3, IMG_SIZE, IMG_SIZE, device=device).to(memory_format=torch.channels_last)
    times = []
    for i in range(n_warmup + n_iters):
        sync()
        t0 = time.perf_counter()
        with autocast():
            model(x)
        sync()
        if i >= n_warmup:
            times.append(time.perf_counter() - t0)
    return 1000 * float(np.median(times)) / batch_size


for name, r in results.items():
    r['params_m'] = count_params(r['model']) / 1e6
    r['gflops'] = count_gflops(r['model'])
    r['lat_bs1'] = measure_latency(r['model'], 1)
    r['lat_bs128'] = measure_latency(r['model'], 128, n_warmup=3, n_iters=10)
    print(f"{SHORT[name]}: {r['params_m']:.1f} M параметров, {r['gflops']:.2f} GFLOPs, "
          f"latency bs=1: {r['lat_bs1']:.2f} мс, bs=128: {r['lat_bs128']:.3f} мс на картинку")

In [ ]:
# --- 1.7 Устойчивость к искажениям ---
rob_x = torch.tensor(test_raw.data[:ROBUST_N]).permute(0, 3, 1, 2).contiguous()
rob_y = torch.tensor(y_test_np[:ROBUST_N])
CORRUPTIONS = {
    'clean': 'без искажений',
    'noise': 'шум sigma=0.08',
    'blur': 'размытие sigma=1',
    'dark': 'затемнение x0.4',
    'low_contrast': 'контраст x0.4',
}


def corrupt(x, kind, gen):
    if kind == 'clean':
        return x
    if kind == 'noise':
        return (x + 0.08 * torch.randn(x.shape, generator=gen, device=x.device)).clamp(0, 1)
    if kind == 'blur':
        return TF.gaussian_blur(x, kernel_size=5, sigma=1.0)
    if kind == 'dark':
        return x * 0.4
    if kind == 'low_contrast':
        m = x.mean(dim=(1, 2, 3), keepdim=True)
        return (x - m) * 0.4 + m
    raise ValueError(kind)


@torch.no_grad()
def eval_corrupted(model, cfg, kind, bs=250):
    model.eval()
    mean = torch.tensor(cfg['mean'], device=device).view(1, 3, 1, 1)
    std = torch.tensor(cfg['std'], device=device).view(1, 3, 1, 1)
    gen = torch.Generator(device=device).manual_seed(SEED)
    correct = 0
    for i in range(0, ROBUST_N, bs):
        x = corrupt(rob_x[i:i + bs].to(device).float() / 255, kind, gen)
        x = F.interpolate(x, size=IMG_SIZE, mode='bilinear', align_corners=False)
        x = ((x - mean) / std).contiguous(memory_format=torch.channels_last)
        with autocast():
            pred = model(x).argmax(1).cpu()
        correct += (pred == rob_y[i:i + bs]).sum().item()
    return correct / ROBUST_N


robust = pd.DataFrame({SHORT[n]: {CORRUPTIONS[k]: eval_corrupted(r['model'], r['cfg'], k) for k in CORRUPTIONS}
                       for n, r in results.items()})
for name, r in results.items():
    col = robust[SHORT[name]]
    r['robust_mean'] = float(col.drop(CORRUPTIONS['clean']).mean())
    r['robust_drop'] = float(col[CORRUPTIONS['clean']] - r['robust_mean'])
print(f"Accuracy на первых {ROBUST_N} тестовых картинках:")
display(robust.round(3))

fig, ax = plt.subplots(figsize=(9, 3.8))
xpos = np.arange(len(robust))
for k, name in enumerate(results):
    ax.bar(xpos + (k - 0.5) * 0.38, robust[SHORT[name]], width=0.36, color=COLORS[name], label=SHORT[name])
ax.set_xticks(xpos)
ax.set_xticklabels(robust.index, fontsize=9)
ax.set_ylabel('accuracy')
ax.set_ylim(0, 1)
ax.set_title('Устойчивость к искажениям')
ax.legend()
plt.show()

In [ ]:
# --- 1.8 Таблица сравнения ---
comparison = pd.DataFrame({
    SHORT[n]: {
        'Предобучение (timm)': r['model'].pretrained_cfg.get('tag', '-'),
        'Параметров, млн': round(r['params_m'], 2),
        'GFLOPs на картинку 224x224': round(r['gflops'], 2),
        'Test accuracy (top-1)': round(r['test_acc'], 4),
        'Test accuracy (top-5)': round(r['test_top5'], 4),
        'Время эпохи, с': round(float(np.mean(r['hist']['epoch_time'])), 1),
        'Latency bs=1, мс': round(r['lat_bs1'], 2),
        'Inference bs=128, мс на картинку': round(r['lat_bs128'], 3),
        'Средняя acc с искажениями': round(r['robust_mean'], 4),
    } for n, r in results.items()
})
display(comparison)

### Выводы по части 1

In [ ]:
r_cnn, r_vit = results['convnext_tiny'], results['vit_small_patch16_224']
better = max(results, key=lambda n: results[n]['test_acc'])
other = [n for n in results if n != better][0]
gap = results[better]['test_acc'] - results[other]['test_acc']
t_cnn, t_vit = np.mean(r_cnn['hist']['epoch_time']), np.mean(r_vit['hist']['epoch_time'])
faster1 = min(results, key=lambda n: results[n]['lat_bs1'])
faster128 = min(results, key=lambda n: results[n]['lat_bs128'])
more_robust = max(results, key=lambda n: results[n]['robust_mean'])
rel = robust.drop(CORRUPTIONS['clean']).sub(robust.loc[CORRUPTIONS['clean']], axis=1)
worst = {SHORT[n]: rel[SHORT[n]].idxmin() for n in results}

text = [
    f"- Test accuracy после {FT_EPOCHS} эпох: {SHORT['convnext_tiny']} - {r_cnn['test_acc']:.1%}, "
    f"{SHORT['vit_small_patch16_224']} - {r_vit['test_acc']:.1%} (top-5: {r_cnn['test_top5']:.1%} и {r_vit['test_top5']:.1%}). "
    + (f"Модели почти на одном уровне, разница {gap * 100:.1f} п.п." if gap < 0.01
       else f"Лучше {SHORT[better]}, разница {gap * 100:.1f} п.п."),
    "- Train и val accuracy на последней эпохе: "
    + '; '.join(f"{SHORT[n]} - {r['hist']['train_acc'][-1]:.3f} и {r['hist']['val_acc'][-1]:.3f}" for n, r in results.items())
    + ". Train accuracy считается с аугментациями и label smoothing, поэтому она может быть ниже val.",
    f"- Параметров у ConvNeXt-T {r_cnn['params_m']:.1f} млн, у ViT-S {r_vit['params_m']:.1f} млн, "
    f"FLOPs {r_cnn['gflops']:.1f} и {r_vit['gflops']:.1f} GFLOPs на картинку 224x224.",
    f"- Эпоха обучения: ConvNeXt-T {t_cnn:.0f} с, ViT-S {t_vit:.0f} с. Latency при bs=1 меньше у {SHORT[faster1]} "
    f"({results[faster1]['lat_bs1']:.2f} мс), при bs=128 на одну картинку быстрее {SHORT[faster128]} "
    f"({results[faster128]['lat_bs128']:.3f} мс). При bs=1 GPU загружен слабо, и время зависит не столько от FLOPs, "
    "сколько от числа и типа операций.",
    f"- С искажениями в среднем лучше держится {SHORT[more_robust]} ({results[more_robust]['robust_mean']:.1%}). "
    "Сильнее всего accuracy падает: " + ', '.join(f"у {k} - {v}" for k, v in worst.items()) + ".",
    "- ConvNeXt - сверточная сеть, в ней заложены локальность и общие веса по всему изображению. ViT режет картинку "
    "на патчи 16x16 и учит связи между ними через attention, поэтому сильнее зависит от предобучения. "
    f"Обе модели предобучены на ImageNet ({r_cnn['model'].pretrained_cfg.get('tag', '-')} и "
    f"{r_vit['model'].pretrained_cfg.get('tag', '-')}), и признаки с ImageNet переносятся на CIFAR-100.",
]
display(Markdown('\n'.join(text)))

In [ ]:
# ============================================================
# ЧАСТЬ 2. Анализ ошибок по классам
# ============================================================
# ЗАДАНИЕ:
# Построить confusion matrix, найти классы, которые модели
# путают чаще всего, визуализировать примеры ошибок.
# ============================================================

# === РЕШЕНИЕ ===

# --- 2.1 Confusion matrix 100x100 для лучшей модели ---
best_name = max(results, key=lambda n: results[n]['test_acc'])
probs, y_true = results[best_name]['probs'], results[best_name]['labels']
y_pred = probs.argmax(1)
err = y_pred != y_true
print(f"Лучшая модель: {SHORT[best_name]}, test accuracy = {1 - err.mean():.4f}, ошибок: {err.sum()} из {len(y_true)}")

cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
order = np.lexsort((np.arange(NUM_CLASSES), FINE_TO_COARSE))
cm_ord = cm[order][:, order]
cm_norm = cm_ord / cm_ord.sum(1, keepdims=True)
cm_err = cm_ord.astype(float)
np.fill_diagonal(cm_err, np.nan)

fig, axes = plt.subplots(1, 2, figsize=(19, 9))
im0 = axes[0].imshow(cm_norm, cmap='Blues', norm=PowerNorm(gamma=0.5, vmin=0, vmax=1))
fig.colorbar(im0, ax=axes[0], shrink=0.7, label='доля от истинного класса')
axes[0].set_title(f'Confusion matrix 100x100, нормирована по строкам ({SHORT[best_name]})')
im1 = axes[1].imshow(cm_err, cmap='Oranges', norm=PowerNorm(gamma=0.5, vmin=0, vmax=np.nanmax(cm_err)))
fig.colorbar(im1, ax=axes[1], shrink=0.7, label='число ошибок')
axes[1].set_title('Только ошибки (без диагонали)')
centers = np.arange(len(COARSE_NAMES)) * 5 + 2
for ax in axes:
    for b in range(5, NUM_CLASSES, 5):
        ax.axhline(b - 0.5, color='0.6', lw=0.4)
        ax.axvline(b - 0.5, color='0.6', lw=0.4)
    ax.set_xticks(centers)
    ax.set_xticklabels(COARSE_NAMES, rotation=90, fontsize=7)
    ax.set_yticks(centers)
    ax.set_yticklabels(COARSE_NAMES, fontsize=7)
    ax.set_xlabel('предсказанный класс (по суперклассам)')
    ax.set_ylabel('истинный класс')
    ax.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
# --- 2.2 Топ-5 пар классов ---
off = cm.copy()
np.fill_diagonal(off, 0)
sym = np.triu(off + off.T, k=1)
flat = np.argsort(sym.ravel())[::-1][:5]
pair_rows = []
for f_idx in flat:
    a, b = divmod(int(f_idx), NUM_CLASSES)
    pair_rows.append({
        'Класс A': CLASSES[a], 'Класс B': CLASSES[b],
        'Ошибок всего': int(sym[a, b]), 'A -> B': int(off[a, b]), 'B -> A': int(off[b, a]),
        'Суперкласс A': COARSE_NAMES[FINE_TO_COARSE[a]], 'Суперкласс B': COARSE_NAMES[FINE_TO_COARSE[b]],
        'Один суперкласс': 'да' if FINE_TO_COARSE[a] == FINE_TO_COARSE[b] else 'нет',
    })
top_pairs = pd.DataFrame(pair_rows, index=pd.RangeIndex(1, 6, name='№'))
display(top_pairs)

print("Топ-5 ошибок по направлению (истина -> прогноз):")
for f_idx in np.argsort(off.ravel())[::-1][:5]:
    a, b = divmod(int(f_idx), NUM_CLASSES)
    print(f"  {CLASSES[a]:>15s} -> {CLASSES[b]:<15s}: {off[a, b]} из 100")

same_super = FINE_TO_COARSE[y_true[err]] == FINE_TO_COARSE[y_pred[err]]
SEMANTIC_SHARE = float(same_super.mean())
CHANCE_SHARE = 4 / (NUM_CLASSES - 1)
print(f"\nОшибок внутри того же суперкласса: {SEMANTIC_SHARE:.1%} (при случайных ошибках {CHANCE_SHARE:.1%})")

In [ ]:
# --- 2.3 Примеры ошибок ---
err_idx = np.where(err)[0]
conf = probs[err_idx, y_pred[err_idx]]
top_err = err_idx[np.argsort(-conf)[:10]]

fig, axes = plt.subplots(2, 5, figsize=(16, 7.2))
for ax, i in zip(axes.ravel(), top_err):
    t, p = y_true[i], y_pred[i]
    ax.imshow(test_raw.data[i], interpolation='bicubic')
    ax.set_title(f"истина: {CLASSES[t]}\nпрогноз: {CLASSES[p]} ({probs[i, p]:.0%})", fontsize=9)
    ax.axis('off')
fig.suptitle(f'10 самых уверенных ошибок {SHORT[best_name]}', y=1.0)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(5, 4, figsize=(10, 13))
for row, f_idx in zip(axes, flat):
    a, b = divmod(int(f_idx), NUM_CLASSES)
    cand = np.where(((y_true == a) & (y_pred == b)) | ((y_true == b) & (y_pred == a)))[0][:4]
    for ax in row:
        ax.axis('off')
    for ax, i in zip(row, cand):
        ax.imshow(test_raw.data[i], interpolation='bicubic')
        ax.set_title(f"{CLASSES[y_true[i]]} -> {CLASSES[y_pred[i]]}", fontsize=9)
fig.suptitle('Примеры для топ-5 пар (истина -> прогноз)', y=1.0)
plt.tight_layout()
plt.show()

In [ ]:
# --- 2.4 Систематические ошибки: яркость, контраст, классы, суперклассы ---
imgs = test_raw.data.astype(np.float32) / 255
gray = imgs @ np.array([0.299, 0.587, 0.114], dtype=np.float32)
brightness = gray.mean(axis=(1, 2))
contrast = gray.std(axis=(1, 2))
saturation = (imgs.max(-1) - imgs.min(-1)).mean(axis=(1, 2))
correct = ~err


def acc_by_quintile(values):
    q = pd.qcut(values, 5, labels=['Q1 (мин)', 'Q2', 'Q3', 'Q4', 'Q5 (макс)'])
    return pd.Series(correct).groupby(q, observed=True).mean()


quint = pd.DataFrame({'Яркость': acc_by_quintile(brightness),
                      'Контраст': acc_by_quintile(contrast),
                      'Насыщенность': acc_by_quintile(saturation)})
print("Accuracy по квинтилям:")
display(quint.round(3))

per_class = pd.DataFrame(classification_report(y_true, y_pred, labels=np.arange(NUM_CLASSES),
                                               target_names=CLASSES, output_dict=True, zero_division=0)).T.iloc[:NUM_CLASSES]
per_class['чаще всего путается с'] = [CLASSES[int(np.argmax(off[c]))] for c in range(NUM_CLASSES)]
per_class['суперкласс'] = [COARSE_NAMES[FINE_TO_COARSE[c]] for c in range(NUM_CLASSES)]
worst_classes = per_class.sort_values('recall').head(10)
print("10 классов с наименьшим recall:")
display(worst_classes[['recall', 'precision', 'f1-score', 'чаще всего путается с', 'суперкласс']].round(2))

super_acc = pd.Series(correct).groupby(FINE_TO_COARSE[y_true]).mean()
super_acc.index = [COARSE_NAMES[i] for i in super_acc.index]
super_acc = super_acc.sort_values()

fig, axes = plt.subplots(1, 2, figsize=(17, 5.5), gridspec_kw={'width_ratios': [1, 1.3]})
xq = np.arange(5)
for k, (col, color) in enumerate(zip(quint.columns, [BLUE, ORANGE, AQUA])):
    axes[0].bar(xq + (k - 1) * 0.27, quint[col], width=0.25, color=color, label=col)
axes[0].set_xticks(xq)
axes[0].set_xticklabels(quint.index)
axes[0].set_ylim(max(0, quint.values.min() - 0.1), 1)
axes[0].axhline(correct.mean(), color='0.4', ls='--', lw=1, label='общая accuracy')
axes[0].set_title('Accuracy по квинтилям яркости, контраста и насыщенности')
axes[0].legend(fontsize=8)
axes[1].barh(super_acc.index, super_acc.values, color=BLUE, height=0.6)
axes[1].axvline(correct.mean(), color='0.4', ls='--', lw=1)
axes[1].set_xlim(max(0, super_acc.min() - 0.1), 1)
axes[1].set_title('Accuracy по суперклассам')
axes[1].tick_params(axis='y', labelsize=8)
plt.tight_layout()
plt.show()

### Выводы по части 2

In [ ]:
def quint_gap(col):
    return quint.loc['Q1 (мин)', col] - quint[col].drop('Q1 (мин)').mean()


def photometry_sentence(col, what):
    g = quint_gap(col)
    if g < -0.03:
        return f"на {what} accuracy ниже, чем на остальных, на {-g * 100:.1f} п.п., это систематическая ошибка."
    if g > 0.03:
        return f"на {what} accuracy даже выше, чем на остальных (на {g * 100:.1f} п.п.)."
    return f"на {what} accuracy почти такая же, как на остальных ({g * 100:+.1f} п.п.), систематической ошибки нет."


n_same_pairs = int((top_pairs['Один суперкласс'] == 'да').sum())
text = [f"- Ошибок у лучшей модели ({SHORT[best_name]}): {err.sum()} из {len(y_true)} тестовых картинок."]
if SEMANTIC_SHARE > 2 * CHANCE_SHARE:
    text.append(f"- {SEMANTIC_SHARE:.0%} ошибок приходится на классы того же суперкласса, а при случайных ошибках "
                f"было бы около {CHANCE_SHARE:.0%}. На confusion matrix это видно по блокам 5x5 у диагонали: "
                "модель чаще всего путает близкие по смыслу классы.")
else:
    text.append(f"- Доля ошибок внутри суперкласса {SEMANTIC_SHARE:.0%}, это близко к случайному уровню "
                f"{CHANCE_SHARE:.0%}, явной смысловой структуры в ошибках нет.")
text.append("- Чаще всего путаются пары: "
            + '; '.join(f"{r['Класс A']} и {r['Класс B']} ({r['Ошибок всего']})" for _, r in top_pairs.iterrows())
            + f". Из них в одном суперклассе {n_same_pairs} из 5.")
text.append("- Худший recall: " + ', '.join(f"{c} ({r['recall']:.2f}, чаще всего путается с {r['чаще всего путается с']})"
                                           for c, r in worst_classes.head(5).iterrows()) + ".")
text.append("- Яркость: " + photometry_sentence('Яркость', 'самых темных картинках (нижний квинтиль)'))
text.append("- Контраст: " + photometry_sentence('Контраст', 'картинках с самым низким контрастом'))
text.append(f"- Хуже всего распознаются суперклассы {', '.join(super_acc.index[:3])}, "
            f"лучше всего - {', '.join(super_acc.index[-3:])}.")
top_conf = probs[top_err, y_pred[top_err]]
text.append(f"- 10 самых уверенных ошибок сделаны с уверенностью от {top_conf.min():.0%} до {top_conf.max():.0%}. "
            "Такие ошибки обычно связаны с неоднозначными картинками или спорной разметкой.")
text.append("- В картинке 32x32 мало деталей, поэтому похожие по форме и фону классы (разные деревья, рыбы, "
            "мелкие животные, люди разного возраста) различить трудно.")
display(Markdown('\n'.join(text)))

In [ ]:
# ============================================================
# ЧАСТЬ 3. Masked Autoencoder (MAE) - самообучение
# ============================================================
# ЗАДАНИЕ:
# Обучить MAE на CIFAR-10 без разметки: маскировать 75% патчей,
# восстанавливать их через ViT-декодер. Затем заморозить encoder
# и обучить линейный классификатор на 1% размеченных данных.
# ============================================================

# === РЕШЕНИЕ ===

for r in results.values():
    r['model'].cpu()
if device.type == 'cuda':
    torch.cuda.empty_cache()

# --- 3.1 Данные целиком на GPU (50000 x 3 x 32 x 32 uint8, около 150 МБ) ---
train_u8 = torch.tensor(train_raw.data).permute(0, 3, 1, 2).contiguous().to(device)
test_u8 = torch.tensor(test_raw.data).permute(0, 3, 1, 2).contiguous().to(device)
CIFAR_MEAN = torch.tensor([0.5071, 0.4865, 0.4409], device=device).view(1, 3, 1, 1)
CIFAR_STD = torch.tensor([0.2673, 0.2564, 0.2762], device=device).view(1, 3, 1, 1)


def gpu_batch(x_u8, augment):
    x = x_u8.float() / 255
    if augment:
        n = x.shape[0]
        x = F.pad(x, (4, 4, 4, 4), mode='reflect')
        ar = torch.arange(32, device=x.device)
        rows = (torch.randint(0, 9, (n, 1), device=x.device) + ar)[:, None, :, None]
        cols = (torch.randint(0, 9, (n, 1), device=x.device) + ar)[:, None, None, :]
        x = x[torch.arange(n, device=x.device)[:, None, None, None],
              torch.arange(3, device=x.device)[None, :, None, None], rows, cols]
        flip = torch.rand(n, device=x.device) < 0.5
        x = torch.where(flip[:, None, None, None], x.flip(3), x)
    x = F.interpolate(x, size=MAE_IMG, mode='bilinear', align_corners=False)
    return (x - CIFAR_MEAN) / CIFAR_STD


def to_display(x):
    return (x * CIFAR_STD + CIFAR_MEAN).clamp(0, 1).permute(0, 2, 3, 1).cpu().numpy()

In [ ]:
# --- 3.2 Упрощенный MAE ---
def sincos_pos_embed_2d(dim, grid_size):
    def emb_1d(pos, d):
        omega = 1.0 / 10000 ** (torch.arange(d // 2, dtype=torch.float32) / (d / 2))
        out = pos.reshape(-1, 1) * omega[None]
        return torch.cat([out.sin(), out.cos()], dim=1)
    g = torch.arange(grid_size, dtype=torch.float32)
    gy, gx = torch.meshgrid(g, g, indexing='ij')
    emb = torch.cat([emb_1d(gy, dim // 2), emb_1d(gx, dim // 2)], dim=1)
    return torch.cat([torch.zeros(1, dim), emb], dim=0).unsqueeze(0)


class PatchEmbed(nn.Module):
    def __init__(self, img_size, patch_size, in_chans, dim):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_chans, dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)


class Attention(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        b, n, c = x.shape
        q, k, v = self.qkv(x).reshape(b, n, 3, self.heads, c // self.heads).permute(2, 0, 3, 1, 4)
        x = F.scaled_dot_product_attention(q, k, v)
        return self.proj(x.transpose(1, 2).reshape(b, n, c))


class Block(nn.Module):
    def __init__(self, dim, heads, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, int(dim * mlp_ratio)), nn.GELU(), nn.Linear(int(dim * mlp_ratio), dim))

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))


def random_masking(x, mask_ratio):
    n, L, d = x.shape
    len_keep = int(L * (1 - mask_ratio))
    noise = torch.rand(n, L, device=x.device)
    ids_shuffle = noise.argsort(dim=1)
    ids_restore = ids_shuffle.argsort(dim=1)
    ids_keep = ids_shuffle[:, :len_keep]
    x_visible = torch.gather(x, 1, ids_keep.unsqueeze(-1).expand(-1, -1, d))
    mask = torch.ones(n, L, device=x.device)
    mask[:, :len_keep] = 0
    mask = torch.gather(mask, 1, ids_restore)
    return x_visible, mask, ids_restore


class MAEEncoder(nn.Module):
    def __init__(self, img_size=MAE_IMG, patch_size=MAE_PATCH, dim=ENC_DIM, depth=ENC_DEPTH, heads=ENC_HEADS):
        super().__init__()
        self.dim = dim
        self.patch_embed = PatchEmbed(img_size, patch_size, 3, dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.register_buffer('pos_embed', sincos_pos_embed_2d(dim, img_size // patch_size))
        self.blocks = nn.ModuleList([Block(dim, heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        nn.init.normal_(self.cls_token, std=0.02)

    def forward(self, x, mask_ratio=0.0):
        x = self.patch_embed(x) + self.pos_embed[:, 1:]
        mask = ids_restore = None
        if mask_ratio > 0:
            x, mask, ids_restore = random_masking(x, mask_ratio)
        cls = (self.cls_token + self.pos_embed[:, :1]).expand(x.shape[0], -1, -1)
        x = torch.cat([cls, x], dim=1)
        for blk in self.blocks:
            x = blk(x)
        return self.norm(x), mask, ids_restore

    def features(self, x):
        z, _, _ = self.forward(x, mask_ratio=0.0)
        return z[:, 1:].mean(dim=1)


class MAE(nn.Module):
    def __init__(self, mask_ratio=MAE_MASK_RATIO):
        super().__init__()
        self.p, self.mask_ratio = MAE_PATCH, mask_ratio
        self.encoder = MAEEncoder()
        self.decoder_embed = nn.Linear(ENC_DIM, DEC_DIM)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, DEC_DIM))
        self.register_buffer('dec_pos_embed', sincos_pos_embed_2d(DEC_DIM, MAE_IMG // MAE_PATCH))
        self.dec_blocks = nn.ModuleList([Block(DEC_DIM, DEC_HEADS) for _ in range(DEC_DEPTH)])
        self.dec_norm = nn.LayerNorm(DEC_DIM)
        self.dec_pred = nn.Linear(DEC_DIM, MAE_PATCH ** 2 * 3)
        nn.init.normal_(self.mask_token, std=0.02)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

    def patchify(self, imgs):
        n, c, h, w = imgs.shape
        g = h // self.p
        x = imgs.reshape(n, c, g, self.p, g, self.p).permute(0, 2, 4, 3, 5, 1)
        return x.reshape(n, g * g, self.p * self.p * c)

    def unpatchify(self, x):
        n, L, _ = x.shape
        g = int(L ** 0.5)
        x = x.reshape(n, g, g, self.p, self.p, 3).permute(0, 5, 1, 3, 2, 4)
        return x.reshape(n, 3, g * self.p, g * self.p)

    def forward(self, imgs, mask_ratio=None):
        mask_ratio = self.mask_ratio if mask_ratio is None else mask_ratio
        latent, mask, ids_restore = self.encoder(imgs, mask_ratio)
        x = self.decoder_embed(latent)
        n, l_vis1, d = x.shape
        L = ids_restore.shape[1]
        mask_tokens = self.mask_token.expand(n, L + 1 - l_vis1, -1)
        x_ = torch.cat([x[:, 1:], mask_tokens], dim=1)
        x_ = torch.gather(x_, 1, ids_restore.unsqueeze(-1).expand(-1, -1, d))
        x = torch.cat([x[:, :1], x_], dim=1) + self.dec_pos_embed
        for blk in self.dec_blocks:
            x = blk(x)
        pred = self.dec_pred(self.dec_norm(x))[:, 1:]
        target = self.patchify(imgs)
        loss = ((pred.float() - target) ** 2).mean(dim=-1)
        loss = (loss * mask).sum() / mask.sum()
        return loss, pred, mask


set_seed(SEED)
mae = MAE().to(device)
print(f"MAE: encoder {count_params(mae.encoder) / 1e6:.2f} M, всего {count_params(mae) / 1e6:.2f} M параметров, "
      f"патчей {mae.encoder.patch_embed.num_patches}, видимых: {int(mae.encoder.patch_embed.num_patches * (1 - MAE_MASK_RATIO))}")

_x = torch.randn(2, 3, MAE_IMG, MAE_IMG, device=device)
assert torch.allclose(mae.unpatchify(mae.patchify(_x)), _x)
_pe = PatchEmbed(MAE_IMG, MAE_PATCH, 3, 4).to(device)
_ref = mae.patchify(_x) @ _pe.proj.weight.permute(0, 2, 3, 1).reshape(4, -1).T + _pe.proj.bias
assert torch.allclose(_pe(_x), _ref, atol=1e-2)
print("проверка patchify/unpatchify пройдена")

In [ ]:
# --- 3.3 Обучение MAE без меток ---
def cosine_with_warmup(step, total, warmup):
    if step < warmup:
        return (step + 1) / warmup
    return 0.5 * (1 + math.cos(math.pi * (step - warmup) / max(1, total - warmup)))


def param_groups(model, wd):
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        (no_decay if p.ndim < 2 or 'token' in n else decay).append(p)
    return [{'params': decay, 'weight_decay': wd}, {'params': no_decay, 'weight_decay': 0.0}]


def train_mae(model):
    ckpt = os.path.join(CKPT_DIR, f'mae_cifar100_{MAE_EPOCHS}ep.pt')
    if os.path.exists(ckpt):
        state = torch.load(ckpt, map_location=device)
        model.load_state_dict(state['model'])
        print(f"MAE: загружен чекпоинт {ckpt}")
        return state['hist']
    steps_per_epoch = len(train_u8) // MAE_BATCH
    total = MAE_EPOCHS * steps_per_epoch
    opt = optim.AdamW(param_groups(model, 0.05), lr=MAE_LR, betas=(0.9, 0.95))
    sched = optim.lr_scheduler.LambdaLR(opt, lambda s: cosine_with_warmup(s, total, MAE_WARMUP * steps_per_epoch))
    scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)
    hist = {'loss': [], 'epoch_time': []}
    for epoch in range(MAE_EPOCHS):
        model.train()
        sync()
        t0 = time.perf_counter()
        perm = torch.randperm(len(train_u8), device=device)
        loss_sum = torch.zeros((), device=device)
        for s in range(steps_per_epoch):
            x = gpu_batch(train_u8[perm[s * MAE_BATCH:(s + 1) * MAE_BATCH]], augment=True)
            with autocast():
                loss, _, _ = model(x)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            loss_sum += loss.detach()
        sync()
        hist['loss'].append(loss_sum.item() / steps_per_epoch)
        hist['epoch_time'].append(time.perf_counter() - t0)
        if epoch == 0 or (epoch + 1) % 5 == 0 or epoch + 1 == MAE_EPOCHS:
            print(f"MAE | эпоха {epoch + 1}/{MAE_EPOCHS} | loss {hist['loss'][-1]:.4f} | {hist['epoch_time'][-1]:.1f} c")
    torch.save({'model': model.state_dict(), 'hist': hist}, ckpt)
    return hist


mae_hist = train_mae(mae)

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(np.arange(1, len(mae_hist['loss']) + 1), mae_hist['loss'], color=BLUE)
ax.set_xlabel('эпоха')
ax.set_ylabel('MSE по скрытым патчам')
ax.set_title('Обучение MAE')
plt.show()

In [ ]:
# --- 3.4 Реконструкция 8 тестовых картинок ---
mae.eval()
torch.manual_seed(SEED)
with torch.no_grad():
    x_vis = gpu_batch(test_u8[:8], augment=False)
    vis_loss, pred, mask = mae(x_vis)
    pred_img = mae.unpatchify(pred.float())
    mask_img = mae.unpatchify(mask.unsqueeze(-1).expand(-1, -1, MAE_PATCH ** 2 * 3))
    gray = (torch.tensor([0.5, 0.5, 0.5], device=device).view(1, 3, 1, 1) - CIFAR_MEAN) / CIFAR_STD
    masked_input = x_vis * (1 - mask_img) + gray * mask_img
    pasted = x_vis * (1 - mask_img) + pred_img * mask_img

rows = [('Оригинал', x_vis), (f'Вход encoder\n({1 - MAE_MASK_RATIO:.0%} патчей)', masked_input),
        ('Реконструкция', pasted), ('Выход decoder\n(все патчи)', pred_img)]
fig, axes = plt.subplots(len(rows), 8, figsize=(16, 8.6))
for r_i, (title, imgs_t) in enumerate(rows):
    for c_i, img in enumerate(to_display(imgs_t)):
        axes[r_i, c_i].imshow(img)
        axes[r_i, c_i].set_xticks([])
        axes[r_i, c_i].set_yticks([])
        axes[r_i, c_i].grid(False)
        if r_i == 0:
            axes[r_i, c_i].set_title(CLASSES[y_test_np[c_i]], fontsize=9)
    axes[r_i, 0].set_ylabel(title, fontsize=9)
fig.suptitle(f'MAE, маскировано {MAE_MASK_RATIO:.0%} патчей, MSE на этих картинках {vis_loss.item():.3f}', y=1.0)
plt.tight_layout()
plt.show()

In [ ]:
# --- 3.5 Linear probing на 1% меток и сравнение с обучением с нуля ---
rng = np.random.default_rng(SEED)
labeled_idx = np.concatenate([rng.choice(np.where(y_train_np == c)[0], LABELED_PER_CLASS, replace=False)
                              for c in range(NUM_CLASSES)])
lab_idx_t = torch.tensor(labeled_idx, device=device)
y_lab = y_train_np[labeled_idx]
y_lab_t = torch.tensor(y_lab, device=device)
y_test_t = torch.tensor(y_test_np, device=device)
print(f"Размеченных картинок: {len(labeled_idx)} ({len(labeled_idx) / len(y_train_np):.1%} train)")


@torch.no_grad()
def extract_features(encoder, x_u8, bs=500):
    encoder.eval()
    out = []
    for i in range(0, len(x_u8), bs):
        with autocast():
            out.append(encoder.features(gpu_batch(x_u8[i:i + bs], augment=False)).float().cpu())
    return torch.cat(out).numpy()


def linear_probe(encoder):
    f_lab = extract_features(encoder, train_u8[lab_idx_t])
    f_test = extract_features(encoder, test_u8)
    clf = make_pipeline(StandardScaler(),
                        LogisticRegressionCV(Cs=[0.01, 0.1, 1.0, 10.0], cv=5, max_iter=3000, n_jobs=-1))
    clf.fit(f_lab, y_lab)
    return float((clf.predict(f_test) == y_test_np).mean())


class EncoderClassifier(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(encoder.dim, NUM_CLASSES)

    def forward(self, x):
        return self.head(self.encoder.features(x))


def train_small_supervised(encoder, seed=SEED):
    set_seed(seed)
    model = EncoderClassifier(encoder).to(device)
    bs = 100
    steps = SMALL_EPOCHS * (len(labeled_idx) // bs)
    opt = optim.AdamW(param_groups(model, 0.05), lr=SMALL_LR)
    sched = optim.lr_scheduler.LambdaLR(opt, lambda s: cosine_with_warmup(s, steps, steps // 10))
    scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    for epoch in range(SMALL_EPOCHS):
        model.train()
        p = torch.randperm(len(labeled_idx), device=device)
        for s in range(len(labeled_idx) // bs):
            sel = p[s * bs:(s + 1) * bs]
            x = gpu_batch(train_u8[lab_idx_t[sel]], augment=True)
            with autocast():
                loss = criterion(model(x), y_lab_t[sel])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, len(test_u8), 500):
            with autocast():
                pred = model(gpu_batch(test_u8[i:i + 500], augment=False)).argmax(1)
            correct += (pred == y_test_t[i:i + 500]).sum().item()
    return correct / len(test_u8)


probe = {}
probe['MAE encoder (заморожен) + линейный классификатор'] = linear_probe(mae.encoder)
set_seed(SEED)
probe['Случайный encoder (заморожен) + линейный классификатор'] = linear_probe(MAEEncoder().to(device))
set_seed(SEED)
probe['С нуля: encoder + голова на 1% меток'] = train_small_supervised(MAEEncoder())
probe['MAE encoder, дообучение целиком на 1% меток'] = train_small_supervised(copy.deepcopy(mae.encoder))
probe['Случайное угадывание'] = 1 / NUM_CLASSES
probe_df = pd.DataFrame({'Test accuracy': probe})
display(probe_df.round(3))

fig, ax = plt.subplots(figsize=(9, 3.6))
labels_short = ['MAE +\nlinear probe', 'случайный encoder\n+ linear probe', 'с нуля', 'MAE +\nдообучение', 'угадывание']
ax.bar(labels_short, probe_df['Test accuracy'], color=[BLUE, '0.6', ORANGE, AQUA, '0.8'], width=0.6)
for k, v in enumerate(probe_df['Test accuracy']):
    ax.text(k, v + 0.003, f'{v:.1%}', ha='center', fontsize=9)
ax.set_ylabel('test accuracy')
ax.set_title('CIFAR-100, 1% меток (5 картинок на класс)')
plt.show()

### Выводы по части 3

In [ ]:
acc_mae = probe['MAE encoder (заморожен) + линейный классификатор']
acc_rand = probe['Случайный encoder (заморожен) + линейный классификатор']
acc_scratch = probe['С нуля: encoder + голова на 1% меток']
acc_ft = probe['MAE encoder, дообучение целиком на 1% меток']


def cmp(a, b):
    d = (a - b) * 100
    if d > 0.5:
        return f"выше на {d:.1f} п.п."
    if d < -0.5:
        return f"ниже на {-d:.1f} п.п."
    return "на том же уровне (разница меньше 0.5 п.п.)."


text = [
    f"- Loss MAE за {len(mae_hist['loss'])} эпох снизился с {mae_hist['loss'][0]:.3f} до {mae_hist['loss'][-1]:.3f}. "
    f"По {1 - MAE_MASK_RATIO:.0%} видимых патчей модель восстанавливает общий вид картинки: цвета, положение объекта "
    "и фона, крупные формы. Мелкие детали размыты, так как при MSE в неясных местах выгоднее предсказывать среднее.",
    f"- Linear probing на 500 размеченных картинках: признаки MAE - {acc_mae:.1%}, случайный замороженный encoder той же "
    f"архитектуры - {acc_rand:.1%}, та же сеть, обученная с нуля, - {acc_scratch:.1%}. "
    f"Linear probe на MAE по сравнению с обучением с нуля {cmp(acc_mae, acc_scratch)} "
    f"По сравнению со случайными признаками {cmp(acc_mae, acc_rand)}",
    f"- Если дообучить MAE encoder целиком на тех же 1% меток, получается {acc_ft:.1%}, по сравнению с обучением "
    f"с нуля {cmp(acc_ft, acc_scratch)}",
]
if acc_mae > acc_scratch + 0.005 or acc_ft > acc_scratch + 0.005:
    text.append("- Предобучение без меток помогло: чтобы восстановить скрытые патчи, encoder должен выучить "
                "структуру изображений, и эти признаки пригодились для классификации при малой разметке.")
else:
    text.append("- В этом запуске предобучение без меток не дало выигрыша над обучением с нуля. Вероятная причина - "
                f"маленький encoder ({count_params(mae.encoder) / 1e6:.1f} млн параметров) и короткое предобучение "
                f"({MAE_EPOCHS} эпох против сотен эпох в статье).")
text.append("- Случайный encoder нужен как ориентир: даже случайные проекции патчей несут информацию о цвете и "
            "расположении, поэтому сравнивать признаки MAE надо с ним, а не с 1% угадывания. Признаки MAE обычно хуже "
            "подходят для линейного классификатора, чем для дообучения всей сети.")
display(Markdown('\n'.join(text)))

In [ ]:
# ============================================================
# ИТОГОВЫЕ ВЫВОДЫ
# ============================================================
text = [
    f"1. ConvNeXt-T и ViT-S после {FT_EPOCHS} эпох дообучения на CIFAR-100: {r_cnn['test_acc']:.1%} и "
    f"{r_vit['test_acc']:.1%} test accuracy. Параметров {r_cnn['params_m']:.1f} и {r_vit['params_m']:.1f} млн, "
    f"FLOPs {r_cnn['gflops']:.1f} и {r_vit['gflops']:.1f} GFLOPs. Latency при bs=1: {r_cnn['lat_bs1']:.2f} и {r_vit['lat_bs1']:.2f} мс, "
    f"средняя accuracy с искажениями: {r_cnn['robust_mean']:.1%} и {r_vit['robust_mean']:.1%}.",
    f"2. У лучшей модели ({SHORT[best_name]}) {SEMANTIC_SHARE:.0%} ошибок внутри одного суперкласса (случайный уровень "
    f"{CHANCE_SHARE:.0%}). Чаще всего путаются "
    + ', '.join(f"{r['Класс A']} и {r['Класс B']}" for _, r in top_pairs.head(3).iterrows()) + ".",
    f"3. MAE без меток: linear probe на 1% меток дает {acc_mae:.1%}, обучение той же сети с нуля - {acc_scratch:.1%}, "
    f"случайный encoder - {acc_rand:.1%}, дообучение MAE encoder - {acc_ft:.1%}.",
    "4. Качество моделей во многом определяется предобучением (с метками на ImageNet в части 1 или без меток в части 3), "
    "а анализ ошибок показывает, какие классы модель путает и почему, чего не видно по одной accuracy.",
]
display(Markdown('\n'.join(text)))